In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, count, date_trunc, md5,
                                  when, current_timestamp, lit)
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "dim_orders")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_orders = spark.read.format("delta").load(f"{silver_base}olist_orders_dataset")

# Transformation logic
df_dim_base = (df_orders.select(
    F.md5(F.col("order_id")).cast("string").alias("order_key"),
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    F.when(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"), lit("On-Time"))
    .when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), lit("Late"))
    .otherwise(lit("Pending/Cancelled")).alias("delivery_status"),
    F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")   
))

# Unknown record
now = datetime.now().replace(microsecond=0)
unknown_data = [("-1", "UNKNOWN", "UNKNOWN", "NA", None, None, None, None, None, "NA", now)]
target_schema = df_dim_base.schema
df_unknown = spark.createDataFrame(unknown_data, schema=target_schema)
df_final = df_dim_base.unionByName(df_unknown)

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    # Merge condition
    data_cols = [c for c in df_final.columns if c not in ["order_key", "gold_load_at"]]
    change_condition = " OR ".join([f"NOT (target.{c} <=> source.{c})" for c in data_cols])

    (dt_gold.alias("target")
     .merge(df_final.alias("source"), "target.order_key = source.order_key")
     .whenMatchedUpdateAll(condition = change_condition)
     .whenNotMatchedInsertAll()
     .execute())
    print(f"--> [Merge] {target_table} updated.")

else:
    #First time load
    df_final.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

    # ZORDERING on order_purchase_timestamp
    spark.sql(f"OPTIMIZE delta.`{gold_path}` ZORDER BY (order_purchase_timestamp)")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")